In [1]:
import cv2 as cv
import mediapipe as mp 
import math
import time

In [4]:
# Initialize MediaPipe Hands module
mphands = mp.solutions.hands
mpdrawing = mp.solutions.drawing_utils

video = cv.VideoCapture("http://172.20.10.5:8080/video")
video.set(cv.CAP_PROP_BUFFERSIZE, 1)

# Get the default frame width and height
frame_width = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create VideoWriter object
fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter('output.mp4', fourcc, 20.0, (frame_width, frame_height))

with mphands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while True:
        # Read frame, check if successful
        ret, frame = video.read() 
        if not ret: break
    
        rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    
        process_frames = hands.process(rgb_frame)

        # Draw landmarks on the frame
        if process_frames.multi_hand_landmarks:
            for lm in process_frames.multi_hand_landmarks:
                mpdrawing.draw_landmarks(frame, lm, mphands.HAND_CONNECTIONS)
                thumb_tip = lm.landmark[4]
                ring_tip = lm.landmark[16]
                pinky_tip = lm.landmark[20]
                index_tip = lm.landmark[8]
                
                # Calculate Euclidean distance

                wrist = lm.landmark[0]
                mid_mcp = lm.landmark[9]
                ref_dist = math.sqrt((wrist.x - mid_mcp.x)**2 + (wrist.y - mid_mcp.y)**2)
                
                distance_1 = math.sqrt((thumb_tip.x - index_tip.x)**2 + (thumb_tip.y - index_tip.y)**2)
                distance_2 = math.sqrt((ring_tip.x - pinky_tip.x)**2 + (ring_tip.y - pinky_tip.y)**2)

                norm_distance_1 = distance_1 / ref_dist
                norm_distance_2 = distance_2 / ref_dist
                
                if (norm_distance_1 > .5) and (norm_distance_2 > .5): 
                    cv.putText(frame, "Using the force", (frame_width//2, 50),cv.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0,255,0), thickness=3)
                
    
        out.write(frame)
        # Display frame
        cv.imshow('Live Video', frame)
            
        # Press 'q' to exit
        if cv.waitKey(1) & 0xFF == ord('q'): 
            break

# Cleanup
out.release()
video.release()
cv.destroyAllWindows()

In [8]:
import cv2 as cv
import mediapipe as mp
import math
import threading
import queue
import socket

def send_gesture(gesture, android_ip="172.20.10.5", port=9000):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect((android_ip, port))
        s.sendall((gesture + "\n").encode())
        s.close()
    except Exception as e:
        print(f"Socket error: {e}")
        
# ---- Threaded frame grabber ----
class FrameGrabber(threading.Thread):
    def __init__(self, url):
        super().__init__(daemon=True)
        self.url = url
        self.frame = None
        self.ret = False
        self.lock = threading.Lock()
        self.cap = cv.VideoCapture(self.url)

    def run(self):
        while True:
            ret, frame = self.cap.read()
            with self.lock:
                self.ret = ret
                self.frame = frame

    def get_latest(self):
        with self.lock:
            return self.ret, self.frame.copy() if self.frame is not None else None

# ---- Setup ----
mphands = mp.solutions.hands
mpdrawing = mp.solutions.drawing_utils

grabber = FrameGrabber("http://localhost:8080/video")
grabber.start()

# Wait for first frame
import time
time.sleep(2)

ret, frame = grabber.get_latest()
frame_height, frame_width = frame.shape[:2]

fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter('output.mp4', fourcc, 20.0, (frame_width, frame_height))

# ---- Main loop ----
with mphands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    while True:
        ret, frame = grabber.get_latest()
        if not ret or frame is None:
            continue

        rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        process_frames = hands.process(rgb_frame)

        if process_frames.multi_hand_landmarks:
            for lm in process_frames.multi_hand_landmarks:
                mpdrawing.draw_landmarks(frame, lm, mphands.HAND_CONNECTIONS)
                thumb_tip = lm.landmark[4]
                ring_tip = lm.landmark[16]
                pinky_tip = lm.landmark[20]
                index_tip = lm.landmark[8]

                wrist = lm.landmark[0]
                mid_mcp = lm.landmark[9]
                ref_dist = math.sqrt((wrist.x - mid_mcp.x)**2 + (wrist.y - mid_mcp.y)**2)

                distance_1 = math.sqrt((thumb_tip.x - index_tip.x)**2 + (thumb_tip.y - index_tip.y)**2)
                distance_2 = math.sqrt((ring_tip.x - pinky_tip.x)**2 + (ring_tip.y - pinky_tip.y)**2)
                norm_distance_1 = distance_1 / ref_dist
                norm_distance_2 = distance_2 / ref_dist

                if (norm_distance_1 > .5) and (norm_distance_2 > .5):
                    cv.putText(frame, "Using the force", (frame_width//2, 50),cv.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0,255,0), thickness=3)
                    send_gesture("FORCE_GESTURE")

        out.write(frame)
        cv.imshow('Live Video', frame)

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

out.release()
cv.destroyAllWindows()

AttributeError: 'NoneType' object has no attribute 'shape'